# Spaceship Titanic — Feature Engineering & Preprocessing

**Notebook 02 of 3** — Build features, impute missing values, and create a preprocessing pipeline.

## Goals

- Engineer new features identified during EDA (Notebook 01)
- Impute missing values using logical rules first, then statistical fallback
- Build an sklearn `Pipeline` and `ColumnTransformer` for reproducible, leak-free preprocessing
- Save processed data for modeling in Notebook 03

## Notebook Structure

1. Setup and Data Loading
2. Feature Engineering
3. Logical Imputation
4. Exploratory Check — Engineered Features
5. Preprocessing Pipeline
6. Train/Validation Split
7. Fit and Transform
8. Save Processed Data

## 1. Setup and Data Loading

**Goal:** Same setup as Notebook 01 — import libraries and load raw data.

**Hints:**
- Same imports as Notebook 01 plus new ones you'll need:
  - `from sklearn.model_selection import train_test_split`
  - `from sklearn.preprocessing import StandardScaler, OneHotEncoder`
  - `from sklearn.impute import SimpleImputer`
  - `from sklearn.compose import ColumnTransformer`
  - `from sklearn.pipeline import Pipeline`
  - `import joblib`
- Load `train.csv` and `test.csv` from `../data/raw/`
- Use the same `RANDOM_STATE = 42` and `target = 'Transported'`
- Verify shapes match what we saw in Notebook 01

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

import joblib

# Local modules
import sys
sys.path.append('../src')
from preprocessor import SpaceshipPreprocessor

RANDOM_STATE = 42
target = 'Transported'

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

In [3]:
print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

Train shape: (8693, 14)
Test shape:  (4277, 13)


## 2. Feature Engineering

**Goal:** Build all the features we identified during EDA.

**Critical rule:** Every feature we engineer on `train` must also be engineered on `test` using the **exact same logic**. A good pattern is to write a function that takes a DataFrame and returns it with new features added, then call it on both.

```python
def engineer_features(df):
    """Apply all feature engineering to a DataFrame.
    Works on both train and test."""
    
    # ... all feature engineering here ...
    
    return df

train = engineer_features(train)
test = engineer_features(test)
```

This guarantees train and test get identical treatment. If you add a feature to one but forget the other, your model will crash at prediction time.

### 2.1 PassengerId Features

**Features to create:** `Group`, `GroupSize`, `IsAlone`

**Hints:**
- `Group` = `PassengerId.str.split('_').str[0]`
- `GroupSize` = use `.groupby('Group')['Group'].transform('count')` — the `transform` method returns a value for every row (unlike `agg` which collapses to one row per group). This is important because we want every passenger to have their group's size, not a separate summary table
- `IsAlone` = `(GroupSize == 1).astype(int)`

**Watch out:** `GroupSize` needs special handling for test data. Groups might span train and test (some family members in train, others in test). For now, compute `GroupSize` independently for train and test — it won't be perfect but it's a reasonable approximation. We can revisit this later if needed.

In [4]:
train["Group"] = train["PassengerId"].str.split("_").str[0]

In [5]:
train["GroupSize"] = train.groupby("Group")["Group"].transform("count")

In [6]:
train["IsAlone"] = (train.GroupSize == 1).astype(int)

### 2.2 Cabin Features

**Features to create:** `Cabin_Deck`, `Cabin_Num`, `Cabin_Side`

**Hints:**
- You already wrote this code in Notebook 01: `.str.split('/').str[0]` etc.
- `Cabin_Num` should be converted to numeric: `.astype(float)` — use float not int because there are NaN values and pandas can't store NaN as int
- For `Cabin_Num_Binned`: use `pd.cut()` or `pd.qcut()` to create Low/Mid/High bins
  - `pd.cut()` creates equal-width bins (0-600, 600-1200, 1200-1800)
  - `pd.qcut()` creates equal-frequency bins (same number of passengers in each bin)
  - `pd.qcut(df['Cabin_Num'], q=3, labels=['Low', 'Mid', 'High'])` is a good starting point

In [7]:
train["Cabin_Deck"] = train["Cabin"].str.split('/').str[0]
train["Cabin_Num"] = train["Cabin"].str.split('/').str[1].astype(float)
train["Cabin_Side"] = train["Cabin"].str.split('/').str[2]

In [8]:
train["Cabin_Num_Binned"] = pd.qcut(train["Cabin_Num"], q = 3, labels = ["Low", "Mid", "High"])

In [9]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,VRDeck,Name,Transported,Group,GroupSize,IsAlone,Cabin_Deck,Cabin_Num,Cabin_Side,Cabin_Num_Binned
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,...,0.0,Maham Ofracculy,False,0001,1,1,B,0.0,P,Low
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,...,44.0,Juanna Vines,True,0002,1,1,F,0.0,S,Low
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,...,49.0,Altark Susent,False,0003,2,0,A,0.0,S,Low
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,...,193.0,Solam Susent,False,0003,2,0,A,0.0,S,Low
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,...,2.0,Willy Santantines,True,0004,1,1,F,1.0,S,Low


### 2.3 Name Features

**Features to create:** `LastName`, `FamilySize`

**Hints:**
- `LastName` = `Name.str.split(' ').str[1]`
- `FamilySize` = same `transform('count')` pattern as GroupSize, but grouped by `LastName`
- Remember from EDA: common surnames were 100% Earth passengers, so this feature is strongest for Earth passengers

**Watch out:** Some `Name` values are NaN. `.str.split()` returns NaN for NaN input, which is fine — but `FamilySize` for NaN LastNames will need handling. Passengers with no name probably shouldn't be grouped together as a "family."

In [10]:
train["LastName"] = train["Name"].str.split(" ").str[1]

In [11]:
train["FirstName"] = train["Name"].str.split(" ").str[0]

In [12]:
train["FamilySize"] = train.groupby("LastName")["LastName"].transform("count")

### 2.4 Age Features

**Feature to create:** `AgeBin`

**Hints:**
- `pd.cut(df['Age'], bins=[0, 12, 17, 25, 65, 200], labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])`
- Note: `pd.cut` excludes the left edge by default, so `bins=[0, 12, ...]` means age 0 falls outside the first bin. Fix this by either setting `include_lowest=True` or starting the first bin at `-1`
- NaN ages will produce NaN AgeBins — that's fine, the imputer will handle it later

In [13]:
train["AgeBin"] = pd.cut(train["Age"], bins = [0, 12, 17, 25, 65, 200],  labels = ['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'], include_lowest = True)

### 2.5 Spending Features

**Features to create:** `TotalSpend`, `LeisureSpend`, `UtilitySpend`, `IsAwakeZeroSpender`

**Hints:**
- `TotalSpend` = sum of all five spending columns. Use `df[spend_cols].sum(axis=1)` — the `axis=1` is critical, it sums across columns for each row rather than down rows for each column
- `LeisureSpend` = `RoomService + Spa + VRDeck`
- `UtilitySpend` = `FoodCourt + ShoppingMall`
- `IsAwakeZeroSpender` = `((df['CryoSleep'] == False) & (df['TotalSpend'] == 0)).astype(int)`

**Watch out:** If any of the five spending columns is NaN for a row, `sum()` will return NaN for TotalSpend (because NaN + anything = NaN). You can handle this with `.fillna(0)` before summing, OR let it stay NaN and let the imputer handle it later. Both are defensible — filling with 0 makes sense because NaN spending probably means $0 spent.

In [14]:
# 2.5 Spending features
# Fill NaN spending with 0 before computing derived features
# NaN spending most likely means $0 spent
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

for col in spend_cols:
    train[col] = train[col].fillna(0)

In [15]:
train["TotalSpend"] = train[spend_cols].sum(axis = 1)

In [16]:
train["LeisureSpend"] = train["RoomService"] + train["Spa"] + train["VRDeck"]

In [17]:
train["UtilitySpend"] = train["FoodCourt"] + train["ShoppingMall"]

In [18]:
train["IsAwakeZeroSpender"] = ((train['CryoSleep'] == False) & (train['TotalSpend'] == 0)).astype(int)

### Feature Engineering Summary

*List all new features created and verify they exist in both train and test. A quick check:*

```python
print(f"Train columns: {train.shape[1]}")
print(f"Test columns:  {test.shape[1]}")
print(f"Column difference: {set(train.columns) - set(test.columns)}")
```

*The only difference should still be `Transported`.*

In [19]:
def engineer_features(df):
    """
    Apply all feature engineering to a DataFrame.
    Works identically on both train and test.
    """
    df = df.copy()  # don't modify the original
    
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    # 2.1 PassengerId features
    df["Group"] = df["PassengerId"].str.split("_").str[0]
    df["GroupSize"] = df.groupby("Group")["Group"].transform("count")
    df["IsAlone"] = (df.GroupSize == 1).astype(int)
    
    # 2.2 Cabin features  
    df["Cabin_Deck"] = df["Cabin"].str.split('/').str[0]
    df["Cabin_Num"] = df["Cabin"].str.split('/').str[1].astype(float)
    df["Cabin_Side"] = df["Cabin"].str.split('/').str[2]
    df["Cabin_Num_Binned"] = pd.qcut(df["Cabin_Num"], q = 3, labels = ["Low", "Mid", "High"])
    
    # 2.3 Name features
    df["LastName"] = df["Name"].str.split(" ").str[1]
    df["FamilySize"] = df.groupby("LastName")["LastName"].transform("count")
    
    # 2.4 Age features
    df["AgeBin"] = pd.cut(df["Age"], bins = [0, 12, 17, 25, 65, 200],  labels = ['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'], include_lowest = True)
    
    # 2.5 Spending features
    # Fill NaN spending with 0 before deriving features
    for col in spend_cols:
        df[col] = df[col].fillna(0)

    df["TotalSpend"] = df[spend_cols].sum(axis=1)
    df["LeisureSpend"] = df["RoomService"] + df["Spa"] + df["VRDeck"]
    df["UtilitySpend"] = df["FoodCourt"] + df["ShoppingMall"]
    df["IsAwakeZeroSpender"] = ((df['CryoSleep'] == False) & (df['TotalSpend'] == 0)).astype(int)

    return df

In [20]:
train_raw = pd.read_csv('../data/raw/train.csv')
test_raw = pd.read_csv('../data/raw/test.csv')

train = engineer_features(train_raw)
test = engineer_features(test_raw)

print(f"Train columns: {train.shape[1]}")
print(f"Test columns:  {test.shape[1]}")
print(f"Column difference: {set(train.columns) - set(test.columns)}")

Train columns: 28
Test columns:  27
Column difference: {'Transported'}


## 3. Logical Imputation

**Goal:** Use domain logic to fill missing values before falling back to statistical imputation.

**Why logical first:** In EDA we discovered that CryoSleep=True passengers always have $0 spending. This means:
- If `CryoSleep` is NaN but all spending columns are 0 → impute `CryoSleep = True`
- If `CryoSleep` is NaN but any spending column > 0 → impute `CryoSleep = False`
- If `CryoSleep` is True but spending is NaN → impute spending as 0

These are more accurate than mode/median imputation because they use real domain constraints.

**Apply to both train and test.** Again, write a function:

```python
def logical_impute(df):
    """Apply domain-logic imputation rules."""
    
    # ... rules here ...
    
    return df
```

**Hints for implementation:**
- To check if all spending is zero: `(df[spend_cols].sum(axis=1) == 0)` — but watch out for NaN. A row with all NaN spending will sum to 0 (if you used `fillna(0)`) or NaN. Think about which case applies to your data
- After logical imputation, check how many NaN values remain with `df.isnull().sum()` — you should see the counts drop
- The remaining NaN values will be handled by `SimpleImputer` in the Pipeline

In [21]:
train[train[spend_cols].sum(axis=1) == 0]

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,Cabin_Num,Cabin_Side,Cabin_Num_Binned,LastName,FamilySize,AgeBin,TotalSpend,LeisureSpend,UtilitySpend,IsAwakeZeroSpender
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,...,0.0,P,Low,Ofracculy,1.0,Adult,0.0,0.0,0.0,1
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,...,0.0,S,Low,Jacostaffey,7.0,Adult,0.0,0.0,0.0,0
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,...,1.0,P,Low,Flatic,3.0,Teen,0.0,0.0,0.0,0
10,0008_02,Europa,True,B/1/P,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,...,1.0,P,Low,Flatic,3.0,Adult,0.0,0.0,0.0,0
18,0016_01,Mars,True,F/5/P,TRAPPIST-1e,45.0,False,0.0,0.0,0.0,...,5.0,P,Low,Upead,3.0,Adult,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8680,9268_01,Earth,True,G/1505/P,TRAPPIST-1e,31.0,False,0.0,0.0,0.0,...,1505.0,P,High,Baldson,6.0,Adult,0.0,0.0,0.0,0
8681,9270_01,Earth,True,G/1497/S,55 Cancri e,33.0,False,0.0,0.0,0.0,...,1497.0,S,High,Mckinsond,10.0,Adult,0.0,0.0,0.0,0
8684,9274_01,NaN,True,G/1508/P,TRAPPIST-1e,23.0,False,0.0,0.0,0.0,...,1508.0,P,High,Bullisey,3.0,YoungAdult,0.0,0.0,0.0,0
8685,9275_01,Europa,False,A/97/P,TRAPPIST-1e,0.0,False,0.0,0.0,0.0,...,97.0,P,Low,Conable,4.0,Child,0.0,0.0,0.0,1


In [22]:
def logical_impute(df):
    df = df.copy()
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    
    # Rule 1: CryoSleep NaN + zero spending → CryoSleep = 1
    cond_null = df['CryoSleep'].isnull()
    cond_zero_spend = df[spend_cols].sum(axis=1) == 0
    rule1 = cond_null & cond_zero_spend
    df.loc[rule1, 'CryoSleep'] = 1

    # Rule 2: CryoSleep NaN + any spending > 0 → CryoSleep = 0
    cond_null = df['CryoSleep'].isnull()
    cond_spend = df[spend_cols].sum(axis=1) > 0
    rule2 = cond_null & cond_spend
    df.loc[rule2, 'CryoSleep'] = 0

    # Rule 3: CryoSleep = 1 + spending NaN → spending = 0
    cond_cryo_true = (df['CryoSleep'] == 1)
    for col in spend_cols:
        df.loc[cond_cryo_true, col] = df.loc[cond_cryo_true, col].fillna(0)

    return df

In [23]:
train = logical_impute(train)
test = logical_impute(test)

In [24]:
print("Missing values after logical imputation:")
print(train.isnull().sum()[train.isnull().sum() > 0])

Missing values after logical imputation:
HomePlanet          201
Cabin               199
Destination         182
Age                 179
VIP                 203
Name                200
Cabin_Deck          199
Cabin_Num           199
Cabin_Side          199
Cabin_Num_Binned    199
LastName            200
FamilySize          200
AgeBin              179
dtype: int64


### Observations — Logical Imputation

- CryoSleep: resolved all 217 missing values using spending rules
  - Passengers with zero spending → imputed CryoSleep = True
  - Passengers with any spending > 0 → imputed CryoSleep = False
- All raw spending columns (RoomService, FoodCourt, etc.): 
  resolved ~110-120 missing values each by filling NaN with 0 
  before computing derived features
- LeisureSpend, UtilitySpend, TotalSpend: 0 missing values 
  after fillna applied inside engineer_features
- Remaining missing values (~179-203 per column) are in 
  demographic and identifier columns with no logical constraints
- These will be handled by SimpleImputer in the Pipeline:
  median for numerical (Age, Cabin_Num), mode for categorical 
  (HomePlanet, Destination, VIP, Cabin_Deck, Cabin_Side, AgeBin)

## 4. Exploratory Check — Engineered Features

**Goal:** Quick sanity check that our new features look reasonable and carry signal.

**Hints:**
- For each new feature, check:
  - `.value_counts()` — does the distribution look reasonable?
  - `.groupby(new_feature)[target].mean()` — does it separate the target?
- Key features to validate:
  - `GroupSize` — does solo travel (GroupSize=1) have a different transport rate than groups?
  - `TotalSpend` — do high spenders get transported at a different rate?
  - `LeisureSpend` vs `UtilitySpend` — does the asymmetry we found in EDA still hold in the engineered features?
  - `AgeBin` — does the child threshold effect show up cleanly?

In [25]:
train["IsAlone"].value_counts()

IsAlone
1    4805
0    3888
Name: count, dtype: int64

In [26]:
train.groupby("IsAlone")[target].mean()

IsAlone
0    0.566872
1    0.452445
Name: Transported, dtype: float64

In [27]:
train["AgeBin"].value_counts()

AgeBin
Adult         4534
YoungAdult    2351
Child          806
Teen           739
Senior          84
Name: count, dtype: int64

In [28]:
train.groupby("AgeBin")[target].mean()

AgeBin
Child         0.699752
Teen          0.553451
YoungAdult    0.458103
Adult         0.484782
Senior        0.476190
Name: Transported, dtype: float64

In [29]:
# Does group size beyond just IsAlone add signal?
train.groupby('GroupSize')[target].mean().sort_index()

GroupSize
1    0.452445
2    0.538050
3    0.593137
4    0.640777
5    0.592453
6    0.614943
7    0.541126
8    0.394231
Name: Transported, dtype: float64

In [30]:
# Does the leisure vs utility distinction hold in the engineered features?
print(train.groupby(pd.cut(train['LeisureSpend'], 
      bins=[-1, 0, 100, 1000, 100000], 
      labels=['Zero', 'Low', 'Mid', 'High']))[target].mean())

LeisureSpend
Zero    0.779103
Low     0.659409
Mid     0.291579
High    0.142077
Name: Transported, dtype: float64


### Observations — Engineered Feature Validation

- `IsAlone`: group travelers transport at 56.7% vs solo at 45.2% — 
  11 point gap confirms feature is predictive. Counterintuitively, 
  group travelers are more likely to be transported
- `AgeBin`: child threshold effect confirmed cleanly — Children 
  transport at 69.9% vs 45-55% for all adult categories. Age only 
  matters as a threshold (child vs adult), not as a continuous value
- `LeisureSpend` binned: near-perfect monotonic relationship with 
  transport rate — Zero (78%) → Low (66%) → Mid (29%) → High (14%). 
  64 point gap between zero and high spenders. Strongest engineered 
  signal in the dataset
- Combined leisure spending is more predictive than any individual 
  spending column alone — confirms feature engineering decision
- Zero leisure spend inflated to 78% by CryoSleep passengers — 
  `IsAwakeZeroSpender` separates this population

## 5. Preprocessing Pipeline

**Goal:** Build a `ColumnTransformer` that handles numerical and categorical columns differently, all within a single pipeline that enforces fit/transform separation.

### Concept Review

**Why Pipeline?** A Pipeline chains preprocessing steps together and ensures that `.fit()` is only ever called on training data. When you call `.transform()` on test data, it uses the statistics (means, modes, category lists) learned from training. This prevents data leakage automatically.

**Why ColumnTransformer?** Different column types need different treatment:
- Numerical columns → impute with median, then scale with StandardScaler
- Categorical columns → impute with most frequent (mode), then OneHotEncode

`ColumnTransformer` lets you define a separate pipeline for each column type and applies them in parallel.

### Step 1: Define column groups

**Hint:** Create lists of columns by type. Think carefully about which columns to include:
- Drop columns that won't be used in modeling: `PassengerId`, `Name`, `Cabin`, `Group`, `LastName` (raw identifiers — we've already extracted features from them)
- Numerical columns: `Age`, spending columns, `TotalSpend`, `LeisureSpend`, `UtilitySpend`, `GroupSize`, `FamilySize`, `Cabin_Num`
- Categorical columns: `HomePlanet`, `Destination`, `Cabin_Deck`, `Cabin_Side`, `Cabin_Num_Binned`, `AgeBin`
- Boolean columns: `CryoSleep`, `VIP` — decide whether to treat as categorical or convert to 0/1 numeric
- Binary engineered features: `IsAlone`, `IsAwakeZeroSpender` — already 0/1, just include as numeric

In [31]:
# Columns to drop entirely — raw identifiers
drop_cols = ['PassengerId', 'Name', 'Cabin', 'Group', 'LastName']

# Numerical columns — impute with median, scale
num_cols = [
    # Demographics
    'Age', 'GroupSize', 'FamilySize', 'Cabin_Num',
    # Boolean flags (already 0/1)
    'CryoSleep', 'VIP', 'IsAlone', 'IsAwakeZeroSpender',
    # Raw spending
    'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
    # Engineered spending
    'TotalSpend', 'LeisureSpend', 'UtilitySpend'
]

# Categorical columns — impute with mode, one-hot encode
cat_cols = [
    'HomePlanet', 'Destination',
    'Cabin_Deck', 'Cabin_Side', 'Cabin_Num_Binned',
    'AgeBin'
]

In [32]:
all_accounted = set(drop_cols + num_cols + cat_cols + [target])
all_columns = set(train.columns)

print(f"Unaccounted columns: {all_columns - all_accounted}")
print(f"Columns in lists but not in train: {all_accounted - all_columns}")

Unaccounted columns: set()
Columns in lists but not in train: set()


### Step 2: Build sub-pipelines

**Hint:**

```python
# Numerical: impute missing with median, then standardize
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: impute missing with mode, then one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])
```

**Important parameters to understand:**
- `drop='first'` in OneHotEncoder: drops one category per feature to avoid the "dummy variable trap" (perfect multicollinearity). For example, if HomePlanet has Earth/Europa/Mars, we only need 2 columns — if both are 0, the model knows it's the third category
- `handle_unknown='ignore'`: if test data contains a category not seen in training, it gets encoded as all zeros instead of throwing an error
- `sparse_output=False`: returns a regular numpy array instead of a sparse matrix — easier to work with

In [33]:
def cast_to_str(X):
    """Convert all columns to string dtype for OHE compatibility."""
    return pd.DataFrame(X).astype(str)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ("to_str", FunctionTransformer(cast_to_str, feature_names_out='one-to-one')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

### Step 3: Combine into ColumnTransformer

**Hint:**

```python
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
], remainder='drop')  # drops any column not listed above
```

`remainder='drop'` is a safety net — any column you didn't explicitly assign to a pipeline gets dropped. This prevents raw identifiers like `PassengerId` from leaking into the model.

In [34]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
], remainder='drop')

## 6. Train/Validation Split

**Goal:** Split the training data into train and validation sets BEFORE fitting the pipeline.

**Why split before fitting?** This gives us a held-out set to evaluate our model on data the pipeline and model have never seen during fitting. If we fit the pipeline on all training data and then split, the pipeline's imputation statistics would include the validation set — a subtle form of leakage.

**Hints:**
- Separate features and target first: `X = train.drop(columns=[target])` and `y = train[target]`
- Use `train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)`
- `stratify=y` ensures both splits have the same proportion of Transported=True/False — important for balanced evaluation
- Name them `X_train, X_val, y_train, y_val` to distinguish from the original `train` DataFrame

In [35]:
X = train.drop(columns=[target])
y = train[target]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [36]:
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"\ny_train transport rate: {y_train.mean():.3f}")
print(f"y_val transport rate:   {y_val.mean():.3f}")

X_train shape: (6954, 27)
X_val shape:   (1739, 27)
y_train shape: (6954,)
y_val shape:   (1739,)

y_train transport rate: 0.504
y_val transport rate:   0.504


## 7. Fit and Transform

**Goal:** Fit the preprocessor on `X_train` only, then transform `X_train`, `X_val`, and `test`.

**Hints:**
- `X_train_processed = preprocessor.fit_transform(X_train)` — fits AND transforms
- `X_val_processed = preprocessor.transform(X_val)` — ONLY transforms (no fitting)
- `X_test_processed = preprocessor.transform(test)` — ONLY transforms

**This is the key moment.** Notice `fit_transform` is called ONCE on `X_train`. Everything else uses `transform` only. This is the entire point of the Pipeline — the statistics (medians, modes, scaling parameters, one-hot category lists) are learned from `X_train` and applied identically everywhere else.

**After transforming, check the output:**
- `X_train_processed.shape` — how many columns do we have now? (OneHotEncoding will have expanded the categoricals)
- Verify no NaN values remain: `np.isnan(X_train_processed).sum()` should be 0

**Getting feature names back:**
- After OneHotEncoding, columns have names like `HomePlanet_Europa`, `Cabin_Deck_B`, etc.
- Get them with: `feature_names = preprocessor.get_feature_names_out()`
- This is essential for SHAP plots later — without names, SHAP just shows `feature_0`, `feature_1`, etc.

In [37]:
# Fit ONLY on X_train, then transform all three sets
X_train_processed =  preprocessor.fit_transform(X_train)  # fit_transform
X_val_processed = preprocessor.transform(X_val)    # transform only
X_test_processed = preprocessor.transform(test)   # transform only

In [38]:
# Check shapes
print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_val_processed shape:   {X_val_processed.shape}")
print(f"X_test_processed shape:  {X_test_processed.shape}")

# Verify no NaN values remain
print(f"\nNaN in X_train: {np.isnan(X_train_processed).sum()}")
print(f"NaN in X_val:   {np.isnan(X_val_processed).sum()}")

# Get OHE category names for categorical columns
ohe = preprocessor.named_transformers_['cat']['encoder']
cat_feature_names = ohe.get_feature_names_out()

# Combine with numerical column names
feature_names = np.array(num_cols + list(cat_feature_names))

print(f"Total features: {len(feature_names)}")
print(feature_names)

X_train_processed shape: (6954, 34)
X_val_processed shape:   (1739, 34)
X_test_processed shape:  (4277, 34)

NaN in X_train: 0
NaN in X_val:   0
Total features: 34
['Age' 'GroupSize' 'FamilySize' 'Cabin_Num' 'CryoSleep' 'VIP' 'IsAlone'
 'IsAwakeZeroSpender' 'RoomService' 'FoodCourt' 'ShoppingMall' 'Spa'
 'VRDeck' 'TotalSpend' 'LeisureSpend' 'UtilitySpend' 'x0_Europa' 'x0_Mars'
 'x1_PSO J318.5-22' 'x1_TRAPPIST-1e' 'x2_B' 'x2_C' 'x2_D' 'x2_E' 'x2_F'
 'x2_G' 'x2_T' 'x3_S' 'x4_Low' 'x4_Mid' 'x5_Child' 'x5_Senior' 'x5_Teen'
 'x5_YoungAdult']


In [39]:
# Build cleaner feature names
ohe = preprocessor.named_transformers_['cat']['encoder']
cat_feature_names = [f"{col}_{val}" for col, cats in 
                     zip(cat_cols, ohe.categories_) 
                     for val in cats[1:]]  # [1:] because drop='first'

feature_names = np.array(num_cols + cat_feature_names)
print(feature_names)

['Age' 'GroupSize' 'FamilySize' 'Cabin_Num' 'CryoSleep' 'VIP' 'IsAlone'
 'IsAwakeZeroSpender' 'RoomService' 'FoodCourt' 'ShoppingMall' 'Spa'
 'VRDeck' 'TotalSpend' 'LeisureSpend' 'UtilitySpend' 'HomePlanet_Europa'
 'HomePlanet_Mars' 'Destination_PSO J318.5-22' 'Destination_TRAPPIST-1e'
 'Cabin_Deck_B' 'Cabin_Deck_C' 'Cabin_Deck_D' 'Cabin_Deck_E'
 'Cabin_Deck_F' 'Cabin_Deck_G' 'Cabin_Deck_T' 'Cabin_Side_S'
 'Cabin_Num_Binned_Low' 'Cabin_Num_Binned_Mid' 'AgeBin_Child'
 'AgeBin_Senior' 'AgeBin_Teen' 'AgeBin_YoungAdult']


In [40]:
# Defined at module level so joblib can pickle it
def cast_to_str(x):
    return x.astype(str)

class SpaceshipPreprocessor:
    """
    Reusable preprocessing pipeline for binary classification projects.
    Fits on training data only and transforms all splits consistently.
    """
    
    def __init__(self, num_cols, cat_cols):
        """Store column configuration and initialize pipeline as None."""
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.preprocessor = None      # will be set after fit
        self.feature_names_ = None    # will be set after fit
    
    def _build_pipeline(self):
        """Build and return the ColumnTransformer pipeline."""
        
        numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
        ])

        categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('to_str', FunctionTransformer(cast_to_str, feature_names_out='one-to-one')),
        ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
        ])

        return ColumnTransformer(transformers=[
        ('num', numeric_pipeline, self.num_cols),
        ('cat', categorical_pipeline, self.cat_cols)
        ], remainder='drop')
    
    def fit_transform(self, X_train, X_val, X_test):
        """
        Fit pipeline on X_train only.
        Transform X_train, X_val, and X_test.
        Returns tuple: (X_train_p, X_val_p, X_test_p)
        """
        # Build and fit the pipeline
        self.preprocessor = self._build_pipeline()
        
        # fit_transform on train, transform only on val and test
        X_train_p = self.preprocessor.fit_transform(X_train)
        X_val_p = self.preprocessor.transform(X_val) 
        X_test_p = self.preprocessor.transform(X_test)
        
        # Build feature names
        self.feature_names_ = self._get_feature_names()
        
        return X_train_p, X_val_p, X_test_p
    
    def _get_feature_names(self):
        """Build readable feature names after fitting."""
        ohe = self.preprocessor.named_transformers_['cat']['encoder']
        cat_feature_names = [f"{col}_{val}" for col, cats in 
                             zip(self.cat_cols, ohe.categories_) 
                             for val in cats[1:]]
        return np.array(self.num_cols + cat_feature_names)
    
    def summary(self, X_train_p, X_val_p, X_test_p):
        """Print shapes, NaN counts, and feature names."""
        print(f"X_train shape: {X_train_p.shape}")
        print(f"X_val shape:   {X_val_p.shape}")
        print(f"X_test shape:  {X_test_p.shape}")
        print(f"\nNaN in X_train: {np.isnan(X_train_p).sum()}")
        print(f"NaN in X_val:   {np.isnan(X_val_p).sum()}")
        print(f"\nTotal features: {len(self.feature_names_)}")
        print(f"Feature names:\n{self.feature_names_}")

In [41]:
sp = SpaceshipPreprocessor(num_cols, cat_cols)
X_train_processed, X_val_processed, X_test_processed = sp.fit_transform(X_train, X_val, test)
sp.summary(X_train_processed, X_val_processed, X_test_processed)
feature_names = sp.feature_names_

X_train shape: (6954, 34)
X_val shape:   (1739, 34)
X_test shape:  (4277, 34)

NaN in X_train: 0
NaN in X_val:   0

Total features: 34
Feature names:
['Age' 'GroupSize' 'FamilySize' 'Cabin_Num' 'CryoSleep' 'VIP' 'IsAlone'
 'IsAwakeZeroSpender' 'RoomService' 'FoodCourt' 'ShoppingMall' 'Spa'
 'VRDeck' 'TotalSpend' 'LeisureSpend' 'UtilitySpend' 'HomePlanet_Europa'
 'HomePlanet_Mars' 'Destination_PSO J318.5-22' 'Destination_TRAPPIST-1e'
 'Cabin_Deck_B' 'Cabin_Deck_C' 'Cabin_Deck_D' 'Cabin_Deck_E'
 'Cabin_Deck_F' 'Cabin_Deck_G' 'Cabin_Deck_T' 'Cabin_Side_S'
 'Cabin_Num_Binned_Low' 'Cabin_Num_Binned_Mid' 'AgeBin_Child'
 'AgeBin_Senior' 'AgeBin_Teen' 'AgeBin_YoungAdult']


### Observations

*How many features do we have after preprocessing? How does that compare to the raw column count? Are there any remaining NaN values?*

## 8. Save Processed Data

**Goal:** Save everything Notebook 03 needs so we don't have to re-run preprocessing.

**Hints:**
- Save the preprocessor itself (for later use on new data):
  - `joblib.dump(preprocessor, '../models/preprocessor.joblib')`
- Save processed arrays and labels:
  - `np.save('../data/processed/X_train.npy', X_train_processed)`
  - `np.save('../data/processed/X_val.npy', X_val_processed)`
  - `np.save('../data/processed/X_test.npy', X_test_processed)`
  - `np.save('../data/processed/y_train.npy', y_train)`
  - `np.save('../data/processed/y_val.npy', y_val)`
- Save feature names for SHAP:
  - `np.save('../data/processed/feature_names.npy', feature_names)`

**Why save?** Notebook 03 can load these directly without re-running all of Notebook 02. This is especially valuable when you're iterating on models — you don't want to re-engineer features every time you tweak a hyperparameter.

In [42]:
# Save preprocessor object
joblib.dump(sp.preprocessor, '../models/preprocessor.joblib')  # ← correct

# Save processed arrays
np.save('../data/processed/X_train.npy', X_train_processed)
np.save('../data/processed/X_val.npy', X_val_processed)
np.save('../data/processed/X_test.npy', X_test_processed)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)

# Save feature names
np.save('../data/processed/feature_names.npy', feature_names)

print("All files saved successfully.")

All files saved successfully.


In [43]:
import os
saved_files = [
    '../models/preprocessor.joblib',
    '../data/processed/X_train.npy',
    '../data/processed/X_val.npy', 
    '../data/processed/X_test.npy',
    '../data/processed/y_train.npy',
    '../data/processed/y_val.npy',
    '../data/processed/feature_names.npy'
]
for f in saved_files:
    exists = os.path.exists(f)
    print(f"{'✓' if exists else '✗'} {f}")

✓ ../models/preprocessor.joblib
✓ ../data/processed/X_train.npy
✓ ../data/processed/X_val.npy
✓ ../data/processed/X_test.npy
✓ ../data/processed/y_train.npy
✓ ../data/processed/y_val.npy
✓ ../data/processed/feature_names.npy


## Summary

### Features Engineered
- **PassengerId:** Group, GroupSize, IsAlone (solo traveler flag)
- **Cabin:** Cabin_Deck, Cabin_Num, Cabin_Num_Binned (Low/Mid/High), Cabin_Side
- **Name:** LastName, FamilySize (surname frequency)
- **Age:** AgeBin (Child/Teen/YoungAdult/Adult/Senior)
- **Spending:** TotalSpend, LeisureSpend, UtilitySpend, IsAwakeZeroSpender
- **Boolean conversion:** CryoSleep and VIP converted from object to 0/1 integer

### Preprocessing Pipeline
- **Numerical columns (16):** SimpleImputer(median) → StandardScaler
- **Categorical columns (6):** SimpleImputer(mode) → cast_to_str → OneHotEncoder(drop='first')
- Pipeline fit on X_train only — transform applied to val and test
- Encapsulated in reusable SpaceshipPreprocessor class in src/preprocessor.py

### Data Sizes
- X_train: (6954, 34)
- X_val:   (1739, 34)
- X_test:  (4277, 34)
- Total features after preprocessing: 34 (16 numerical + 18 from OHE)

### Ready for Notebook 03
All processed data and fitted preprocessor saved to disk.
Notebook 03 loads these directly and trains Logistic Regression,
Random Forest, and XGBoost classifiers.